In [1]:
import sys
from pathlib import Path

parent_dir = Path.cwd().parent if 'jupyter_book_tutorial' in str(Path.cwd()) else Path.cwd()
if str(parent_dir) not in sys.path:
    sys.path.insert(0, str(parent_dir))

from bluesky import RunEngine
from ophyd import Signal
from bluesky.plans import scan
from bluesky_config.devices import MockDetector
from bluesky_config.metadata import create_experiment_metadata

# We'll capture documents in a list to examine them
captured_docs = []

def document_collector(name, doc):
    """Callback to collect all documents."""
    captured_docs.append((name, doc))

# Setup
RE = RunEngine({})
RE.subscribe(document_collector)

motor = Signal(name='motor', value=0)
detector = MockDetector(name='det')

# Run a small scan
uid = RE(scan([detector], motor, 0, 5, 5))

print(f"✓ Scan complete. Captured {len(captured_docs)} documents.")

✓ Scan complete. Captured 8 documents.

In [2]:
import json

# Get start document
start_name, start_doc = captured_docs[0]

print(f"Document type: {start_name}")
print(f"\nKey fields in start document:")
print(json.dumps({
    'uid': start_doc['uid'][:16] + '...',
    'time': start_doc['time'],
    'plan_name': start_doc['plan_name'],
    'plan_args': start_doc.get('plan_args', {}),
    'scan_id': start_doc.get('scan_id'),
}, indent=2))

Document type: start


Key fields in start document:

{
  "uid": "f10b3ff0-56cb-46...",
  "time": 1760888945.126902,
  "plan_name": "scan",
  "plan_args": {
    "detectors": [
      "MockDetector(prefix='', name='det', read_attrs=['value'], configuration_attrs=[])"
    ],
    "num": 5,
    "args": [
      "Signal(name='motor', value=0, timestamp=1760888945.12399)",
      0,
      5
    ],
    "per_step": "None"
  },
  "scan_id": 1
}

In [3]:
# Get descriptor document
desc_name, desc_doc = captured_docs[1]

print(f"Document type: {desc_name}")
print(f"\nData keys (what will be measured):")
for key, info in desc_doc['data_keys'].items():
    print(f"  - {key}: {info['dtype']} (shape: {info['shape']})")

Document type: descriptor


Data keys (what will be measured):

  - det_value: number (shape: [])

  - motor: number (shape: [])

In [4]:
# Get event documents (skip start and descriptor)
events = [(name, doc) for name, doc in captured_docs[2:-1] if name == 'event']

print(f"Total events: {len(events)}")
print(f"\nFirst event data:")
first_event = events[0][1]
print(json.dumps({
    'seq_num': first_event.get('seq_num'),
    'data': first_event['data'],
    'timestamps': first_event['timestamps']
}, indent=2))

print(f"\nLast event data:")
last_event = events[-1][1]
print(json.dumps({
    'seq_num': last_event.get('seq_num'),
    'data': last_event['data'],
}, indent=2))

Total events: 5


First event data:

{
  "seq_num": 1,
  "data": {
    "det_value": 0.9680503559738911,
    "motor": 0.0
  },
  "timestamps": {
    "det_value": 1760888945.130099,
    "motor": 1760888945.128549
  }
}


Last event data:

{
  "seq_num": 5,
  "data": {
    "det_value": 1.0180583543607562,
    "motor": 5.0
  }
}

In [5]:
# Get stop document
stop_name, stop_doc = captured_docs[-1]

print(f"Document type: {stop_name}")
print(f"\nStop document:")
print(json.dumps({
    'run_start': stop_doc['run_start'][:16] + '...',
    'time': stop_doc['time'],
    'exit_status': stop_doc['exit_status'],
    'num_events': stop_doc.get('num_events', {})
}, indent=2))

Document type: stop


Stop document:

{
  "run_start": "f10b3ff0-56cb-46...",
  "time": 1760888945.1414652,
  "exit_status": "success",
  "num_events": {
    "primary": 5
  }
}

In [6]:
# Check for document files
doc_dir = parent_dir / 'data' / 'documents'
doc_files = list(doc_dir.glob('*.jsonl'))

if doc_files:
    latest_file = max(doc_files, key=lambda p: p.stat().st_mtime)
    print(f"Latest document file: {latest_file.name}")
    print(f"Size: {latest_file.stat().st_size} bytes")
    print(f"\nFirst 5 lines:")
    
    with open(latest_file, 'r') as f:
        for i, line in enumerate(f):
            if i >= 5:
                break
            doc = json.loads(line)
            doc_type = list(doc.keys())[0]
            print(f"  Line {i+1}: {doc_type} document")
else:
    print("No document files found yet.")

Latest document file: 7fe914bb_documents.jsonl

Size: 17676 bytes


First 5 lines:

  Line 1: start document

  Line 2: descriptor document

  Line 3: event document

  Line 4: event document

  Line 5: event document

In [7]:
# Show all captured data as a table
import pandas as pd

# Extract data from events
data_rows = []
for name, doc in captured_docs:
    if name == 'event':
        row = doc['data'].copy()
        row['seq_num'] = doc.get('seq_num', 0)
        data_rows.append(row)

df = pd.DataFrame(data_rows)
df = df[['seq_num', 'motor', 'det_value']]  # Reorder columns

print("Captured data as table:")
print(df)

Captured data as table:

   seq_num  motor  det_value
0        1   0.00   0.968050
1        2   1.25   1.057992
2        3   2.50   1.019279
3        4   3.75   0.989766
4        5   5.00   1.018058